# Chapter 13 &mdash; A Transformer on a Language That Needs a Tape

**Concept 14 of the Chapter 13 decomposition:** *A Transformer on a Language That Needs a Tape*

Karpathy's baby GPT trained on $w\#w$ &mdash; the language this chapter built a DTM for, with that DTM marking the homework.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Karpathy-GPT-On-A-Jove-TM/Concept-Karpathy-GPT-On-A-Jove-TM.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


This is the third and last run of Karpathy's baby GPT, and the language is the one
this chapter is about.

* Chapter 6 gave it a **regular** language, and it did well.
* Chapter 12 gave it a **context-free** language, and it did badly.
* Here it gets $w\#w$, which is not even context-free.

Two things are different from the earlier notebooks, and both are worth reading before
the code.

**The corpus is generated, not filtered.** Strings of the form $w\#w$ are
exponentially rare among all strings over $\{0,1,\#\}$, so enumerating and testing
would find almost nothing. Every $w$ is written down and doubled instead. The
vocabulary is three tokens now, not two.

**The corpus is balanced across lengths.** Take every word up to length 6 and there
are 32 times as many length-6 words as length-1 words; every per-length measurement
would then be computed from a handful of positions, and a handful of positions is how
you get a 100% that is really one lucky guess. So the same number of words is taken at
each length, and the counts are printed under the percentages.

The DTM of Concept 9 is the judge &mdash; after being checked itself.

## 2. Definitions

### The model &mdash; Karpathy's, unchanged

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

### Cutting a sequence into examples, and the training loop

In [ ]:
def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

### The machine of Concept 9

In [ ]:
# --- the DTM for w#w, the cross-off machine of Concept 9 -----------------
# Mark the first unmarked symbol on the left (0 becomes X, 1 becomes Y),
# walk right past the #, mark the matching one on the right, walk home.
# F has no outgoing transitions, so reaching it is halting in F.
WSW = md2mc('''TM
I  : X ; X , R -> I
I  : Y ; Y , R -> I
I  : 0 ; X , R -> R0
I  : 1 ; Y , R -> R1
I  : # ; # , R -> Ck

R0 : 0 ; 0 , R -> R0
R0 : 1 ; 1 , R -> R0
R0 : # ; # , R -> S0
S0 : X ; X , R -> S0
S0 : Y ; Y , R -> S0
S0 : 0 ; X , L -> Lb

R1 : 0 ; 0 , R -> R1
R1 : 1 ; 1 , R -> R1
R1 : # ; # , R -> S1
S1 : X ; X , R -> S1
S1 : Y ; Y , R -> S1
S1 : 1 ; Y , L -> Lb

Lb : 0 ; 0 , L -> Lb
Lb : 1 ; 1 , L -> Lb
Lb : X ; X , L -> Lb
Lb : Y ; Y , L -> Lb
Lb : # ; # , L -> Lb
Lb : . ; . , R -> I

Ck : X ; X , R -> Ck
Ck : Y ; Y , R -> Ck
Ck : . ; . , S -> F
''')

### The corpus

In [ ]:
# --- the corpus.  GENERATED, not filtered --------------------------------
# Strings of the form w#w are exponentially rare, so the enumerate-and-test
# recipe of Chapters 6 and 12 would find almost nothing.  Write the w down
# and double it instead.
#
# `per` words of EVERY length matters more than it looks.  Take all of them
# and the length-6 words outnumber the length-1 words 32 to 1, so every
# per-length number below would be computed from a handful of positions --
# that is how you get a 100% that is one lucky guess.  Balance first.
import random
from itertools import product

SYM = {'0': 0, '1': 1, '#': 2}

def corpus(nmax=6, per=64, seed=0, mirror=False):
    rnd, out = random.Random(seed), []
    for n in range(1, nmax + 1):
        ws = [''.join(w) for w in product('01', repeat=n)]
        for i in range(per):
            w = ws[i % len(ws)]
            out.append(w + '#' + (w[::-1] if mirror else w))
    rnd.shuffle(out)
    return out

def encode(strings):
    """the flat token sequence, plus for each position which second-half
       symbol it is: its |w| and its offset j into the second half."""
    seq, wlen, pos = [], [], []
    for s in strings:
        n = s.index('#')
        for i, c in enumerate(s):
            seq.append(SYM[c])
            wlen.append(n if i > n else None)
            pos.append(i - n - 1 if i > n else None)
    return seq, wlen, pos

### Measuring, and one training run

In [ ]:
# --- accuracy on the second half, grouped by whatever you like -----------
# The model is asked for the single most likely next token (argmax, not a
# sample), and it is asked only at positions that COPY -- the second half.
# 50% is the coin: the two bits are equally likely when you cannot see the
# original.
def accuracy_by(gpt, k, seq, tag):
    idx = [i for i in range(k, len(seq)) if tag[i] is not None]
    hit, tot = {}, {}
    for b in range(0, len(idx), 8192):
        chunk = idx[b:b + 8192]
        X = torch.tensor([seq[i - k:i] for i in chunk], dtype=torch.long)
        pred = gpt(X).argmax(-1).tolist()
        for i, p in zip(chunk, pred):
            t = tag[i]
            tot[t] = tot.get(t, 0) + 1
            hit[t] = hit.get(t, 0) + (p == seq[i])
    return {t: (hit[t] / tot[t], tot[t]) for t in sorted(tot)}

def table(rows, keys, label):
    print('%-9s' % label + ''.join('%7s' % k for k in keys))
    for name, acc in rows:
        print('%-9s' % name + ''.join('%6.0f%%' % (100 * acc[k][0])
                                      for k in keys))
    print('%-9s' % 'samples' + ''.join('%7d' % rows[0][1][k][1] for k in keys))

# --- one training run, with every knob in the signature ------------------
def train_at(seq, k, iters=300, n_embd=16, seed=1337, quiet=False):
    X, Y = make_XY(seq, k)
    config = GPTConfig(block_size=k, vocab_size=3, n_layer=4, n_head=4,
                       n_embd=n_embd, bias=False)
    torch.manual_seed(seed)
    gpt = GPT(config)
    losses = train_gpt(gpt, X, Y, iters=iters,
                       every=iters if quiet else iters // 4)
    return gpt, losses[-1]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;13.&nbsp;The Compact ID Notation $aqb$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Compact-ID-Notation/Concept-Compact-ID-Notation.ipynb) &nbsp;&middot;&nbsp; [**Chapter 13** index](https://github.com/ganeshutah/Jove/blob/master/Chapter13-TM/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;15.&nbsp;The Context Window Is the Memory](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Context-Window-Is-The-Memory/Concept-Context-Window-Is-The-Memory.ipynb)&nbsp;&rarr;

---

## 3. Tests

The judge first. A TM halts when no transition applies, and accepts if it halts in a final state.

In [ ]:
def tm_accepts(T, tape, fuel=4000):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

bad = []
for n in range(5):
    for a in product('01', repeat=n):
        for m in range(5):
            for b in product('01', repeat=m):
                s = ''.join(a) + '#' + ''.join(b)
                if tm_accepts(WSW, s) != (a == b):
                    bad.append(s)
print('disagreements with a direct string comparison :', bad)
assert not bad
print('the DTM is exactly w#w -- it can be trusted as judge')

Watch it cross off one pair, so the machine is not a black box.

In [ ]:
print("running the DTM on '011#011' :")
trunc, halts = run_tm(WSW, '011#011', 4000, chatty=False)
for cfg, path in halts:
    print('   halted in state %s with tape %s' % (cfg[0], cfg[2].rstrip('.')))
print()
print("X marks a crossed-off 0 and Y a crossed-off 1; every symbol got")
print("paired with the one |w|+1 places to its right.")

**The corpus.** Same number of words at every length.

In [ ]:
strings = corpus(nmax=6, per=64)
seq, wlen, pos = encode(strings)
print('strings :', len(strings))
print('symbols :', len(seq))
print('example :', strings[0])
print()
import collections
print('words per length :',
      sorted(collections.Counter(s.index('#') for s in strings).items()))

Cut it up. Three tokens in the vocabulary now, so the coin sits at 1/3 for a uniform guess &mdash; but on the second half, where only `0` and `1` can occur, a model that knows that much is already at 1/2.

In [ ]:
context_length = 6
X, Y = make_XY(seq, context_length)
for i in range(6):
    print('example %2d: %s --> %s' % (i + 1, X[i].tolist(), Y[i].item()))
print(X.shape, Y.shape)

**Train.**

In [ ]:
gpt, final = train_at(seq, context_length, iters=300)

**Now the question this notebook exists for.** Not the loss &mdash; the accuracy on exactly those symbols that have to be copied, grouped by the length of the word being copied.

In [ ]:
acc = accuracy_by(gpt, context_length, seq, wlen)
table([('k=%d' % context_length, acc)], list(range(1, 7)), '|w| =')

There is a wall in that row, and it is not where impatience would put it.

In [ ]:
print("The accuracy does not trail off gently.  It holds up, and then it is")
print("at 50%, which is the coin exactly.")
print()
print("Look at WHERE the wall is, and compare it with the window size 6.")
print()
print("The next notebook derives that position on paper before measuring it,")
print("and then shows that no amount of training moves it.")

Finally, watch it copy. At each position of the second half the model is given the TRUE symbols before it &mdash; the same conditions the table above measured &mdash; and asked for the next one. The lengths matter: 5 fits inside a window of 6, and 6 does not.

In [ ]:
def detok(ids):
    return ''.join('01#'[i] for i in ids)

print('%-14s %-16s %-5s %s' % ('word', 'model copied it as', '|w|', 'right'))
for w in ('01101', '11010', '010101', '110011'):
    true_ids = [SYM[c] for c in w + '#' + w]
    guess = []
    for j in range(len(w)):
        cut = len(w) + 1 + j                    # the position being predicted
        x = torch.tensor(true_ids[cut - context_length:cut],
                         dtype=torch.long)[None, ...]
        guess.append(int(torch.argmax(gpt(x)[0])))
    g = detok(guess)
    print('%-14s %-16s %-5d %d/%d'
          % (w, g, len(w), sum(a == b for a, b in zip(w, g)), len(w)))
print()
print('The two |w|=5 words are mostly copied; the two |w|=6 words are not.')
print()
print('Let the model run on its OWN output instead of the true prefix and')
print('it does worse still, because one wrong symbol poisons the window')
print('that the next prediction reads.  Try it: that is exercise 6.')
print()
print('The DTM of Concept 9 copies all four exactly, and would copy a word')
print('of length 500.  That is what having a tape buys.')

## 4. Animation

The machine of Concept 9, doing the job the model is trying to imitate.

In [ ]:
from jove.AnimateTM import *
AnimateTM(WSW, FuseEdges=True)

## 5. Exercises


1. Where is the wall, and what is `context_length`? Write down the relationship you
   think you see before reading the next notebook.
2. Raise `per` from 64 to 128. The percentages should barely move and the sample
   counts should double. If a percentage *does* move a lot, what does that tell you
   about the one you had before?
3. Set `per=8` and re-run. Some columns will look dramatic. Explain why you should not
   believe them.
4. The corpus is shuffled before being run together. What would go wrong if it were
   left sorted by length?
5. The vocabulary has three tokens but the second half only ever contains two. How
   much of the loss is the model spending on learning where `#` goes?
6. Rewrite the copying cell so each prediction is appended to the input and the model
   reads its OWN previous guesses instead of the true ones. The scores get worse.
   Explain the mechanism in one sentence, and say which of the two numbers is the fair
   one to quote for "has it learned to copy".

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter13-TM/Concept-Karpathy-GPT-On-A-Jove-TM')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')